[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/D.%20Strategic%20Network%20Design%20%26%20Sourcing%20%28The%20Blueprint%20of%20the%20Business%29/087.%20The%20Hub-and-Spoke%20Network%20Design%20Problem/P87-Tier-5_executed.ipynb)

---


# 87. The Hub-and-Spoke Network Design Problem

## Tier 5: Integrated Digital-Twin Simulation

### Purpose and scope

This notebook builds a small, self-contained simulation of a hub-and-spoke network monitoring workflow. It generates synthetic port, weather, vessel, and demand data; updates a simulated network state; applies rule-based hub reconfiguration; explores what-if scenarios; and summarizes modeled performance indicators.

The code is an educational prototype, not a production digital twin. It does not connect to live IoT sensors, weather services, vessel-tracking systems, or operational databases. It does not train or validate a predictive model, and its reported KPIs are simplified indicators rather than empirical measurements. In particular, the reconfiguration rule and its stated expected improvement are illustrative heuristics, not the result of a mathematical optimization model or a measured causal effect.

### Modeling assumptions

- The network contains 15 ports, with three initial hubs.
- Port coordinates are synthetic locations in broad Pacific regions. Distances used for hub-to-hub comparisons are great-circle distances on a spherical Earth, in kilometers.
- Demand, utilization, weather, vessel positions, and disruption events are generated synthetically. The random-number generators are not seeded, so separate runs can produce different outcomes.
- The simulated clock advances in 30-minute increments. One simulation run covers 24 hours.
- KPI formulas, capacity effects, scenario costs, and recommendations are simplified teaching examples. They should not be interpreted as calibrated operational forecasts.

### Workflow

1. Generate the synthetic network and initialize its monitoring components.
2. Collect simulated observations and compute network KPIs at each update.
3. Periodically apply a threshold-based rule to decide whether to reconfigure hubs.
4. Record the resulting state and decisions, then visualize the simulation.
5. Evaluate separate what-if scenarios and summarize the simulated history.

### How to interpret the results

The outputs show how data collection, state updates, rule-based decisions, historical records, and scenario analysis can be connected in one workflow. They do not establish that the simulated policy improves a real network. Since the data are stochastic and no seed is set, exact values and decisions can vary from run to run.

In [ ]:
from copy import deepcopy
from datetime import datetime, timedelta
import random
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

## 1. Build the simulation components

A **digital-twin workflow** keeps a software representation of a system updated from observations and uses that representation to support analysis. Here, the observations are generated by the notebook itself, so the model demonstrates the workflow structure rather than a live physical-to-virtual connection.

At each update, the simulator creates a state containing port utilization, weather conditions, vessel positions, demand forecasts, disruption alerts, and KPIs. The rule-based optimizer inspects this state and may select a new set of hubs. Its score and improvement estimate are illustrative; they are not computed by minimizing an explicitly defined network cost function.

In [ ]:
class DemandForecaster:
    """Generate synthetic hourly demand values between pairs of ports."""

    def __init__(self, num_ports: int):
        self.num_ports = num_ports
        self.base_demand = np.random.randint(100, 500, (num_ports, num_ports))
        np.fill_diagonal(self.base_demand, 0)

    def generate_forecast(self) -> np.ndarray:
        """Return a synthetic 24-hour demand forecast."""
        forecast = np.zeros((self.num_ports, self.num_ports, 24))
        for hour in range(24):
            hourly_factor = 1 + 0.3 * np.sin(2 * np.pi * hour / 24)
            noise = np.random.normal(1, 0.1, (self.num_ports, self.num_ports))
            forecast[:, :, hour] = self.base_demand * hourly_factor * noise
            np.fill_diagonal(forecast[:, :, hour], 0)
        return forecast


class DisruptionDetector:
    """Flag ports with high modeled storm probability or utilization."""

    def __init__(self, num_ports: int):
        self.num_ports = num_ports

    def detect_disruptions(
        self,
        weather_conditions: Dict[int, dict],
        port_utilization: np.ndarray,
        timestamp: datetime,
    ) -> List[dict]:
        disruptions = []
        for port_id in range(self.num_ports):
            weather = weather_conditions[port_id]
            utilization = port_utilization[port_id]

            if weather["storm_probability"] > 0.3:
                disruptions.append({
                    "type": "weather",
                    "port": port_id,
                    "severity": "high" if weather["storm_probability"] > 0.6 else "medium",
                    "description": f"Storm probability: {weather['storm_probability']:.2f}",
                    "timestamp": timestamp,
                })

            if utilization > 0.9:
                disruptions.append({
                    "type": "capacity",
                    "port": port_id,
                    "severity": "critical" if utilization > 0.95 else "high",
                    "description": f"High utilization: {utilization:.2f}",
                    "timestamp": timestamp,
                })
        return disruptions


class RealTimeOptimizer:
    """Apply an illustrative threshold rule to suggest a hub configuration."""

    def __init__(self, num_ports: int):
        self.num_ports = num_ports

    def optimize(self, network_state: dict, current_hubs: List[int], disruptions: List[dict]) -> dict:
        reason = ""
        for disruption in disruptions:
            if (
                disruption["port"] in current_hubs
                and disruption["severity"] in {"high", "critical"}
            ):
                reason = (
                    f"Disruption at Hub {disruption['port'] + 1}: "
                    f"{disruption['description']}"
                )
                break

        average_utilization = float(np.mean(network_state["port_utilization"]))
        if average_utilization > 0.85:
            reason = f"High network utilization: {average_utilization:.2f}"

        if reason:
            new_hubs = self._generate_new_hub_configuration(network_state, disruptions)
            # This random estimate illustrates a recommendation field; it is not a measured benefit.
            expected_improvement = random.uniform(0.05, 0.15)
            return {
                "action": "reconfigure",
                "old_hub_configuration": current_hubs.copy(),
                "new_hub_configuration": new_hubs,
                "expected_improvement": expected_improvement,
                "reason": reason,
                "implementation_time": "30 minutes",
            }

        return {
            "action": "maintain",
            "old_hub_configuration": current_hubs.copy(),
            "new_hub_configuration": current_hubs.copy(),
            "expected_improvement": 0.0,
            "reason": "No reconfiguration threshold was triggered",
            "implementation_time": "N/A",
        }

    def _generate_new_hub_configuration(self, network_state: dict, disruptions: List[dict]) -> List[int]:
        disrupted_ports = {
            event["port"]
            for event in disruptions
            if event["severity"] in {"high", "critical"}
        }
        available_ports = [
            port for port in range(self.num_ports) if port not in disrupted_ports
        ]

        if len(available_ports) >= 3:
            scores = []
            for port in available_ports:
                utilization = network_state["port_utilization"][port]
                score = (1 - utilization) * 0.7 + random.uniform(0, 0.3)
                scores.append((port, score))
            scores.sort(key=lambda item: item[1], reverse=True)
            return [port for port, _ in scores[:3]]

        return available_ports[:3]


class HubSpokeDigitalTwin:
    """Coordinate synthetic data, state updates, scenarios, and reporting."""

    def __init__(self, num_ports: int = 15, update_interval_minutes: int = 30):
        if num_ports < 3:
            raise ValueError("The simulation requires at least three ports.")
        self.num_ports = num_ports
        self.update_interval = update_interval_minutes
        self.current_time = datetime.now()
        self.port_coordinates = self._generate_port_coordinates()
        self.hub_ports = [2, 7, 12]
        self.port_capacities = np.random.randint(8000, 15000, num_ports)
        self.port_costs = np.random.randint(50000, 150000, num_ports)

        self.iot_sensors = self._initialize_iot_sensors()
        self.weather_stations = self._initialize_weather_stations()
        self.vessel_tracking = self._initialize_vessel_tracking()
        self.demand_forecaster = DemandForecaster(num_ports)
        self.disruption_detector = DisruptionDetector(num_ports)
        self.optimization_engine = RealTimeOptimizer(num_ports)

        self.current_network_state = {
            "port_utilization": np.zeros(num_ports),
            "vessel_positions": {},
            "weather_conditions": {},
            "demand_forecasts": np.zeros((num_ports, num_ports, 24)),
            "disruption_alerts": [],
            "performance_kpis": {},
        }
        self.historical_data = {
            "timestamps": [],
            "network_states": [],
            "optimization_decisions": [],
            "performance_metrics": [],
        }

    def _generate_port_coordinates(self) -> np.ndarray:
        """Create synthetic longitude-latitude locations for three regions."""
        coordinates = []
        for _ in range(5):
            coordinates.append([random.uniform(-125, -115), random.uniform(32, 49)])
        for _ in range(5):
            coordinates.append([random.uniform(-180, -140), random.uniform(-20, 20)])
        for _ in range(5):
            coordinates.append([random.uniform(120, 145), random.uniform(20, 45)])
        return np.asarray(coordinates[:self.num_ports], dtype=float)

    def _initialize_iot_sensors(self) -> Dict[int, dict]:
        """Create synthetic sensor inventories; no live devices are contacted."""
        sensors = {}
        for port_id in range(self.num_ports):
            sensors[port_id] = {
                "container_sensors": random.randint(20, 50),
                "crane_sensors": random.randint(5, 15),
                "gate_sensors": random.randint(10, 25),
                "environmental_sensors": random.randint(5, 10),
                "status": "active",
                "last_update": self.current_time,
            }
        return sensors

    def _initialize_weather_stations(self) -> Dict[int, dict]:
        """Create synthetic weather observations for each port."""
        stations = {}
        for port_id in range(self.num_ports):
            stations[port_id] = {
                "wind_speed": random.uniform(0, 25),
                "wave_height": random.uniform(0, 5),
                "visibility": random.uniform(1, 10),
                "temperature": random.uniform(10, 35),
                "pressure": random.uniform(980, 1030),
                "storm_probability": random.uniform(0, 0.3),
                "last_update": self.current_time,
            }
        return stations

    def _initialize_vessel_tracking(self) -> Dict[int, dict]:
        """Create synthetic vessel records with initial positions and destinations."""
        vessels = {}
        for vessel_id in range(random.randint(30, 50)):
            vessels[vessel_id] = {
                "name": f"Vessel_{vessel_id + 1}",
                "type": random.choice(["Container", "Bulk", "Tanker"]),
                "capacity": random.randint(1000, 8000),
                "current_position": self._generate_random_position(),
                "destination": random.randrange(self.num_ports),
                "eta": self.current_time + timedelta(hours=random.randint(6, 72)),
                "status": random.choice(["in_transit", "at_port", "loading", "unloading"]),
                "speed": random.uniform(10, 25),
                "last_update": self.current_time,
            }
        return vessels

    def _generate_random_position(self) -> dict:
        """Return a synthetic position in latitude-longitude coordinates."""
        return {
            "latitude": random.uniform(-50, 60),
            "longitude": random.uniform(-180, 180),
        }

    def collect_real_time_data(self) -> None:
        """Generate one synthetic observation batch and update the network state."""
        utilization = self.current_network_state["port_utilization"]
        for port_id in range(self.num_ports):
            utilization[port_id] = random.uniform(0.4, 0.95)
            weather = self.weather_stations[port_id]
            weather["wind_speed"] = max(0, weather["wind_speed"] + random.uniform(-2, 2))
            weather["wave_height"] = max(0, weather["wave_height"] + random.uniform(-0.5, 0.5))
            weather["storm_probability"] = max(
                0, min(1, weather["storm_probability"] + random.uniform(-0.05, 0.05))
            )
            weather["last_update"] = self.current_time
            self.current_network_state["weather_conditions"][port_id] = weather

        for vessel_id, vessel in self.vessel_tracking.items():
            if vessel["status"] == "in_transit":
                vessel["current_position"] = self._update_vessel_position(vessel)
                vessel["last_update"] = self.current_time
            self.current_network_state["vessel_positions"][vessel_id] = vessel

        self.current_network_state["demand_forecasts"] = (
            self.demand_forecaster.generate_forecast()
        )
        self.current_network_state["disruption_alerts"] = (
            self.disruption_detector.detect_disruptions(
                self.current_network_state["weather_conditions"],
                utilization,
                self.current_time,
            )
        )
        self.current_network_state["performance_kpis"] = self._calculate_kpis()

    def _update_vessel_position(self, vessel: dict) -> dict:
        """Apply a small random displacement as a simple movement illustration."""
        position = vessel["current_position"]
        scale = vessel["speed"] / 20
        return {
            "latitude": position["latitude"] + random.uniform(-0.5, 0.5) * scale,
            "longitude": position["longitude"] + random.uniform(-0.5, 0.5) * scale,
        }

    def _calculate_kpis(self) -> dict:
        """Compute simplified indicators from the current synthetic state."""
        return {
            "average_utilization": float(np.mean(self.current_network_state["port_utilization"])),
            "total_vessels": len(self.vessel_tracking),
            "vessels_at_port": sum(
                vessel["status"] == "at_port" for vessel in self.vessel_tracking.values()
            ),
            "active_disruptions": len(self.current_network_state["disruption_alerts"]),
            "network_efficiency": self._calculate_network_efficiency(),
            "service_level": self._calculate_service_level(),
            "cost_per_container": self._calculate_cost_per_container(),
            "carbon_footprint": self._calculate_carbon_footprint(),
        }

    def _calculate_network_efficiency(self) -> float:
        """Combine mean utilization and a simplified weather penalty."""
        utilization_efficiency = float(np.mean(self.current_network_state["port_utilization"]))
        weather_impact = sum(
            weather["storm_probability"] * 0.1
            for weather in self.current_network_state["weather_conditions"].values()
            if weather["storm_probability"] > 0.2
        ) / self.num_ports
        return max(0.0, min(1.0, utilization_efficiency * (1 - weather_impact)))

    def _calculate_service_level(self) -> float:
        """Estimate service level using utilization and disruption penalties."""
        mean_utilization = float(np.mean(self.current_network_state["port_utilization"]))
        utilization_penalty = max(0.0, mean_utilization - 0.8) * 0.5
        disruption_penalty = min(
            0.2, len(self.current_network_state["disruption_alerts"]) * 0.05
        )
        return max(0.0, min(1.0, 0.95 - utilization_penalty - disruption_penalty))

    def _calculate_cost_per_container(self) -> float:
        """Return a simplified modeled cost indicator in dollars per container."""
        mean_utilization = float(np.mean(self.current_network_state["port_utilization"]))
        weather_surcharge = sum(
            weather["storm_probability"]
            for weather in self.current_network_state["weather_conditions"].values()
        ) * 10
        return 150 * (1 + mean_utilization * 0.3) + weather_surcharge / self.num_ports

    def _calculate_carbon_footprint(self) -> float:
        """Return a simplified emissions indicator in modeled tons of CO2."""
        vessel_emissions = len(self.vessel_tracking) * 50
        port_emissions = float(np.sum(self.current_network_state["port_utilization"])) * 100
        return vessel_emissions + port_emissions

    def run_real_time_optimization(self) -> dict:
        """Apply the threshold rule and record accepted reconfiguration decisions."""
        recommendations = self.optimization_engine.optimize(
            self.current_network_state,
            self.hub_ports,
            self.current_network_state["disruption_alerts"],
        )
        if recommendations["expected_improvement"] > 0.05:
            self.hub_ports = recommendations["new_hub_configuration"]
            self.historical_data["optimization_decisions"].append({
                "timestamp": self.current_time,
                "type": "hub_reconfiguration",
                "old_hubs": recommendations["old_hub_configuration"],
                "new_hubs": recommendations["new_hub_configuration"],
                "expected_improvement": recommendations["expected_improvement"],
                "reason": recommendations["reason"],
            })
        return recommendations

    def simulate_what_if_scenarios(self) -> dict:
        """Evaluate four independent, synthetic planning scenarios."""
        return {
            "typhoon_disruption": self._simulate_typhoon_scenario(),
            "demand_surge": self._simulate_demand_surge(),
            "new_hub_addition": self._simulate_new_hub(),
            "capacity_expansion": self._simulate_capacity_expansion(),
        }

    def _simulate_typhoon_scenario(self) -> dict:
        """Estimate a simplified capacity impact at a randomly selected hub."""
        affected_hub = random.choice(self.hub_ports)
        capacity_reduction = random.uniform(0.3, 0.7)
        duration_hours = random.randint(24, 72)
        original_capacity = self.port_capacities[affected_hub]
        alternative_hubs = [hub for hub in self.hub_ports if hub != affected_hub]
        alternative_hub = (
            min(alternative_hubs, key=lambda hub: self._calculate_distance(affected_hub, hub))
            if alternative_hubs else None
        )
        return {
            "scenario_type": "typhoon_disruption",
            "affected_hub": affected_hub,
            "capacity_reduction": capacity_reduction,
            "duration_hours": duration_hours,
            "original_capacity": original_capacity,
            "reduced_capacity": original_capacity * (1 - capacity_reduction),
            "alternative_hub": alternative_hub,
            "estimated_cost_impact": capacity_reduction * original_capacity * 0.1,
            "recommended_actions": self._generate_typhoon_response(
                affected_hub, alternative_hub
            ),
        }

    def _simulate_demand_surge(self) -> dict:
        """Estimate mean utilization under a randomly generated demand increase."""
        demand_increase = random.uniform(1.2, 1.5)
        affected_regions = random.sample(
            range(self.num_ports), random.randint(3, min(7, self.num_ports))
        )
        current_utilization = float(np.mean(self.current_network_state["port_utilization"]))
        projected_utilization = min(1.0, current_utilization * demand_increase)
        return {
            "scenario_type": "demand_surge",
            "demand_increase": demand_increase,
            "affected_regions": affected_regions,
            "current_utilization": current_utilization,
            "projected_utilization": projected_utilization,
            "capacity_risk": projected_utilization > 0.9,
            "recommended_actions": self._generate_demand_surge_response(demand_increase),
        }

    def _simulate_new_hub(self) -> dict:
        """Rank non-hub ports using the notebook's illustrative candidate score."""
        non_hubs = [port for port in range(self.num_ports) if port not in self.hub_ports]
        if not non_hubs:
            return {"scenario_type": "new_hub_addition", "status": "no_candidates"}
        best_candidate = max(non_hubs, key=self._evaluate_hub_candidate)
        current_efficiency = self.current_network_state["performance_kpis"]["network_efficiency"]
        projected_efficiency = min(1.0, current_efficiency * 1.1)
        return {
            "scenario_type": "new_hub_addition",
            "candidate_hub": best_candidate,
            "candidate_score": self._evaluate_hub_candidate(best_candidate),
            "current_efficiency": current_efficiency,
            "projected_efficiency": projected_efficiency,
            "implementation_cost": self.port_costs[best_candidate],
            "payback_period_months": self._calculate_payback_period(best_candidate),
            "recommended": projected_efficiency - current_efficiency > 0.05,
        }

    def _simulate_capacity_expansion(self) -> dict:
        """Compare three illustrative expansion levels at the busiest current hub."""
        utilization_by_hub = {
            hub: self.current_network_state["port_utilization"][hub]
            for hub in self.hub_ports
        }
        if not utilization_by_hub:
            return {"scenario_type": "capacity_expansion", "status": "no_hubs"}
        target_hub = max(utilization_by_hub, key=utilization_by_hub.get)
        current_capacity = self.port_capacities[target_hub]
        current_utilization = utilization_by_hub[target_hub]
        results = []
        for expansion in [0.2, 0.3, 0.5]:
            new_capacity = current_capacity * (1 + expansion)
            expansion_cost = new_capacity * 50
            new_utilization = current_utilization / (1 + expansion)
            efficiency_gain = (current_utilization - new_utilization) * 0.5
            results.append({
                "expansion_percentage": expansion,
                "new_capacity": new_capacity,
                "expansion_cost": expansion_cost,
                "current_utilization": current_utilization,
                "projected_utilization": new_utilization,
                "efficiency_gain": efficiency_gain,
                "roi": efficiency_gain * 10000 / expansion_cost if expansion_cost > 0 else 0,
            })
        return {
            "scenario_type": "capacity_expansion",
            "target_hub": target_hub,
            "current_capacity": current_capacity,
            "current_utilization": current_utilization,
            "expansion_options": results,
            "recommended_option": max(results, key=lambda option: option["roi"]),
        }

    def _calculate_distance(self, port1: int, port2: int) -> float:
        """Calculate great-circle distance between port coordinates, in kilometers."""
        lon1, lat1 = np.radians(self.port_coordinates[port1])
        lon2, lat2 = np.radians(self.port_coordinates[port2])
        delta_lat = lat2 - lat1
        delta_lon = lon2 - lon1
        haversine = (
            np.sin(delta_lat / 2) ** 2
            + np.cos(lat1) * np.cos(lat2) * np.sin(delta_lon / 2) ** 2
        )
        central_angle = 2 * np.arcsin(np.sqrt(np.clip(haversine, 0, 1)))
        return float(6371.0 * central_angle)

    def _evaluate_hub_candidate(self, port_id: int) -> float:
        """Score a candidate using centrality, capacity, cost, and utilization."""
        centrality_score = self._calculate_centrality(port_id)
        capacity_score = self.port_capacities[port_id] / np.max(self.port_capacities)
        cost_score = 1 - self.port_costs[port_id] / np.max(self.port_costs)
        utilization_score = 1 - self.current_network_state["port_utilization"][port_id]
        return (
            centrality_score * 0.3
            + capacity_score * 0.3
            + cost_score * 0.2
            + utilization_score * 0.2
        )

    def _calculate_centrality(self, port_id: int) -> float:
        """Use inverse mean great-circle distance as a simple centrality score."""
        distances = [
            self._calculate_distance(port_id, other)
            for other in range(self.num_ports)
            if other != port_id
        ]
        return 1 / (1 + float(np.mean(distances))) if distances else 0.5

    def _calculate_payback_period(self, hub_id: int) -> float:
        """Return the illustrative investment-to-monthly-benefit ratio."""
        investment = self.port_costs[hub_id]
        monthly_benefit = investment * 0.05
        return investment / monthly_benefit if monthly_benefit > 0 else float("inf")

    def _generate_typhoon_response(
        self, affected_hub: int, alternative_hub: Optional[int]
    ) -> List[str]:
        actions = [
            f"Activate the emergency response protocol for Hub {affected_hub + 1}",
            f"Reroute {random.randint(20, 40)}% of traffic to alternative hubs",
        ]
        if alternative_hub is not None:
            actions.append(f"Review available capacity at Hub {alternative_hub + 1}")
        actions.extend([
            "Notify affected customers of potential delays",
            "Review contingency vessel contracts",
            "Update delivery-time estimates",
        ])
        return actions

    def _generate_demand_surge_response(self, demand_increase: float) -> List[str]:
        actions = [
            f"Review surge capacity for a modeled {(demand_increase - 1):.0%} increase",
            "Review operating hours at high-utilization ports",
            "Assess standby vessel availability",
        ]
        if demand_increase > 1.3:
            actions.append("Evaluate temporary hub activation")
        actions.extend([
            "Consider priority routing for critical cargo",
            "Coordinate capacity planning with alliance partners",
        ])
        return actions

    def update_historical_data(self) -> None:
        """Store snapshots so later state updates do not alter recorded history."""
        self.historical_data["timestamps"].append(self.current_time)
        self.historical_data["network_states"].append(deepcopy(self.current_network_state))
        self.historical_data["performance_metrics"].append(
            deepcopy(self.current_network_state["performance_kpis"])
        )
        if len(self.historical_data["timestamps"]) > 1000:
            for key in self.historical_data:
                self.historical_data[key] = self.historical_data[key][-1000:]


print("Initializing the synthetic hub-and-spoke simulation...")
digital_twin = HubSpokeDigitalTwin(num_ports=15, update_interval_minutes=30)
print(f"Number of ports: {digital_twin.num_ports}")
print(f"Initial hub ports: {[hub + 1 for hub in digital_twin.hub_ports]}")
print(f"Update interval: {digital_twin.update_interval} minutes")
print(f"Ports with sensor inventories: {len(digital_twin.iot_sensors)}")
print(f"Weather records: {len(digital_twin.weather_stations)}")
print(f"Synthetic vessel records: {len(digital_twin.vessel_tracking)}")

## 2. Simulate 24 hours of network updates

Each iteration advances the simulated clock, generates a new synthetic observation batch, and recomputes the KPIs. The threshold rule is evaluated every four updates, corresponding to two simulated hours. Historical snapshots are recorded after each update so that the trend plots can use the full run.

In [ ]:
simulation_hours = 24
updates_per_hour = 60 // digital_twin.update_interval
total_updates = simulation_hours * updates_per_hour

print(f"Running {simulation_hours} simulated hours ({total_updates} updates)...")
for update_cycle in range(total_updates):
    digital_twin.current_time += timedelta(minutes=digital_twin.update_interval)
    digital_twin.collect_real_time_data()

    if update_cycle % 4 == 0:
        optimization_result = digital_twin.run_real_time_optimization()
        if optimization_result["action"] == "reconfigure":
            print(f"[{digital_twin.current_time:%H:%M}] HUB RECONFIGURATION")
            print(f"  Reason: {optimization_result['reason']}")
            print(
                "  Hubs: "
                f"{[hub + 1 for hub in optimization_result['old_hub_configuration']]} -> "
                f"{[hub + 1 for hub in optimization_result['new_hub_configuration']]}"
            )
            print(
                "  Illustrative expected-improvement estimate: "
                f"{optimization_result['expected_improvement']:.1%}"
            )

    digital_twin.update_historical_data()

    if update_cycle % 8 == 0:
        kpis = digital_twin.current_network_state["performance_kpis"]
        disruptions = digital_twin.current_network_state["disruption_alerts"]
        print(f"[{digital_twin.current_time:%H:%M}] NETWORK STATUS")
        print(f"  Modeled network efficiency: {kpis['network_efficiency']:.1%}")
        print(f"  Modeled service level: {kpis['service_level']:.1%}")
        print(f"  Vessels recorded as at port: {kpis['vessels_at_port']}/{kpis['total_vessels']}")
        print(f"  Active modeled disruptions: {len(disruptions)}")
        print(f"  Modeled cost per container: ${kpis['cost_per_container']:.2f}")

final_kpis = digital_twin.current_network_state["performance_kpis"]
print("\nSIMULATION COMPLETE")
print(f"Current hubs: {[hub + 1 for hub in digital_twin.hub_ports]}")
print(f"Modeled network efficiency: {final_kpis['network_efficiency']:.1%}")
print(f"Modeled service level: {final_kpis['service_level']:.1%}")
print(f"Average port utilization: {final_kpis['average_utilization']:.1%}")
print(f"Modeled carbon indicator: {final_kpis['carbon_footprint']:.0f} tons CO2")
print(
    "Recorded hub reconfigurations: "
    f"{len(digital_twin.historical_data['optimization_decisions'])}"
)

## 3. Visualize the simulated network state

The dashboard combines four views: port locations and hub status, current KPI values, disruption counts, and recent historical trends. Port markers are colored by utilization; the color bar gives the utilization scale. Hub-to-hub lines show the connections among the currently selected hubs, not a complete shipping-route plan.

In [ ]:
def visualize_digital_twin_dashboard(digital_twin: HubSpokeDigitalTwin):
    """Plot the current synthetic network state and its recent KPI history."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle("Hub-and-Spoke Simulation Dashboard", fontsize=16, fontweight="bold")

    ax1.set_title("Port Locations and Hub Status", fontweight="bold")
    utilization = digital_twin.current_network_state["port_utilization"]
    spoke_points = [
        port for port in range(digital_twin.num_ports)
        if port not in digital_twin.hub_ports
    ]
    if spoke_points:
        scatter = ax1.scatter(
            digital_twin.port_coordinates[spoke_points, 0],
            digital_twin.port_coordinates[spoke_points, 1],
            c=utilization[spoke_points], cmap="RdYlGn_r", vmin=0, vmax=1,
            s=150, edgecolors="black", label="Non-hub port",
        )
        colorbar = fig.colorbar(scatter, ax=ax1, fraction=0.046, pad=0.04)
        colorbar.set_label("Port utilization (fraction)")
    hub_coords = digital_twin.port_coordinates[digital_twin.hub_ports]
    ax1.scatter(
        hub_coords[:, 0], hub_coords[:, 1], s=300, c="red", marker="s",
        edgecolors="black", linewidths=1.5, label="Hub",
    )
    for port, (longitude, latitude) in enumerate(digital_twin.port_coordinates):
        ax1.text(longitude + 0.2, latitude + 0.2, str(port + 1), fontsize=8)
    for index, hub1 in enumerate(digital_twin.hub_ports):
        for hub2 in digital_twin.hub_ports[index + 1:]:
            points = digital_twin.port_coordinates[[hub1, hub2]]
            ax1.plot(points[:, 0], points[:, 1], color="blue", linewidth=1.5, alpha=0.6)
    ax1.set_xlabel("Longitude (degrees)")
    ax1.set_ylabel("Latitude (degrees)")
    ax1.grid(True, alpha=0.3)
    ax1.legend()

    ax2.set_title("Current Modeled Performance Indicators", fontweight="bold")
    kpis = digital_twin.current_network_state["performance_kpis"]
    kpi_names = ["Efficiency", "Service level", "Mean utilization", "Cost / $500"]
    normalized_values = [
        kpis["network_efficiency"],
        kpis["service_level"],
        kpis["average_utilization"],
        kpis["cost_per_container"] / 500,
    ]
    bars = ax2.bar(kpi_names, normalized_values, color=["#FF6B6B", "#4ECDC4", "#FFD93D", "#95E77E"])
    ax2.set_ylabel("Displayed value (cost scaled by $500)")
    ax2.set_ylim(0, 1)
    ax2.tick_params(axis="x", rotation=15)
    ax2.grid(True, axis="y", alpha=0.3)
    labels = [
        f"{kpis['network_efficiency']:.1%}",
        f"{kpis['service_level']:.1%}",
        f"{kpis['average_utilization']:.1%}",
        f"${kpis['cost_per_container']:.0f}",
    ]
    for bar, label in zip(bars, labels):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                 label, ha="center", va="bottom", fontsize=9)

    ax3.set_title("Current Modeled Disruptions", fontweight="bold")
    disruptions = digital_twin.current_network_state["disruption_alerts"]
    severity_counts = {"weather": 0, "capacity": 0}
    for event in disruptions:
        severity_counts[event["type"]] += 1
    if disruptions:
        bars = ax3.bar(
            list(severity_counts), list(severity_counts.values()),
            color=["#FF6B6B", "#FFD93D"],
        )
        ax3.set_ylabel("Number of alerts")
        ax3.grid(True, axis="y", alpha=0.3)
        for bar, count in zip(bars, severity_counts.values()):
            ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.1,
                     str(count), ha="center", va="bottom")
    else:
        ax3.text(0.5, 0.5, "No modeled disruptions", ha="center", va="center",
                 transform=ax3.transAxes, fontsize=13)
        ax3.set_axis_off()

    ax4.set_title("Recent Simulated KPI History", fontweight="bold")
    metrics = digital_twin.historical_data["performance_metrics"][-24:]
    if len(metrics) > 1:
        efficiency = [item["network_efficiency"] for item in metrics]
        service = [item["service_level"] for item in metrics]
        elapsed_hours = np.arange(len(metrics)) * digital_twin.update_interval / 60
        ax4.plot(elapsed_hours, efficiency, "o-", label="Efficiency", color="#FF6B6B")
        ax4.plot(elapsed_hours, service, "s-", label="Service level", color="#4ECDC4")
        ax4.set_xlabel("Elapsed time within displayed history (hours)")
        ax4.set_ylabel("Fraction")
        ax4.set_ylim(0, 1)
        ax4.legend()
        ax4.grid(True, alpha=0.3)
    else:
        ax4.text(0.5, 0.5, "Insufficient historical data", ha="center", va="center",
                 transform=ax4.transAxes)

    fig.tight_layout()
    plt.show()
    return fig


fig = visualize_digital_twin_dashboard(digital_twin)

## 4. Evaluate what-if scenarios

Each scenario is a separate synthetic calculation based on the current simulation state. The scenarios are not forecasts: they use randomly chosen inputs or simplified arithmetic to make the consequences and assumptions visible. A recommendation from one scenario should therefore be read as a prompt for further analysis, not as an operational instruction.

In [ ]:
scenarios = digital_twin.simulate_what_if_scenarios()
print("WHAT-IF SCENARIO ANALYSIS")

for scenario_name, scenario_data in scenarios.items():
    print(f"\n{scenario_name.replace('_', ' ').title()}:")

    if scenario_data["scenario_type"] == "typhoon_disruption":
        alternative = scenario_data["alternative_hub"]
        alternative_label = str(alternative + 1) if alternative is not None else "N/A"
        print(f"  Affected hub: {scenario_data['affected_hub'] + 1}")
        print(f"  Modeled capacity reduction: {scenario_data['capacity_reduction']:.1%}")
        print(f"  Assumed duration: {scenario_data['duration_hours']} hours")
        print(f"  Illustrative cost impact: ${scenario_data['estimated_cost_impact']:,.0f}")
        print(f"  Nearest alternative hub: {alternative_label}")
        print(f"  Suggested actions listed: {len(scenario_data['recommended_actions'])}")

    elif scenario_data["scenario_type"] == "demand_surge":
        print(f"  Modeled demand multiplier: {scenario_data['demand_increase']:.1%}")
        print(f"  Affected regions: {len(scenario_data['affected_regions'])}")
        print(f"  Current mean utilization: {scenario_data['current_utilization']:.1%}")
        print(f"  Projected mean utilization: {scenario_data['projected_utilization']:.1%}")
        print(f"  Modeled capacity risk: {'Yes' if scenario_data['capacity_risk'] else 'No'}")

    elif scenario_data["scenario_type"] == "new_hub_addition":
        if "candidate_hub" in scenario_data:
            print(f"  Candidate hub: {scenario_data['candidate_hub'] + 1}")
            print(f"  Illustrative candidate score: {scenario_data['candidate_score']:.4f}")
            print(f"  Current modeled efficiency: {scenario_data['current_efficiency']:.1%}")
            print(f"  Assumed projected efficiency: {scenario_data['projected_efficiency']:.1%}")
            print(f"  Illustrative implementation cost: ${scenario_data['implementation_cost']:,.0f}")
            print(f"  Assumed payback period: {scenario_data['payback_period_months']:.1f} months")
            print(f"  Passes the scenario threshold: {'Yes' if scenario_data['recommended'] else 'No'}")
        else:
            print(f"  Status: {scenario_data['status']}")

    elif scenario_data["scenario_type"] == "capacity_expansion":
        if "target_hub" in scenario_data:
            print(f"  Target hub: {scenario_data['target_hub'] + 1}")
            print(f"  Current modeled capacity: {scenario_data['current_capacity']:,.0f}")
            print(f"  Current utilization: {scenario_data['current_utilization']:.1%}")
            print(f"  Expansion options evaluated: {len(scenario_data['expansion_options'])}")
            option = scenario_data["recommended_option"]
            print(f"  Highest illustrative ROI option: {option['expansion_percentage']:.0%} expansion")
            print(f"  Illustrative ROI: {option['roi']:.4f}")
            print(f"  Modeled new capacity: {option['new_capacity']:,.0f}")
        else:
            print(f"  Status: {scenario_data['status']}")

## 5. Visualize scenario comparisons

The charts use separate units in the typhoon panel, so its bars are intended for within-category inspection rather than direct comparison across categories. The demand-surge and hub-addition panels compare modeled current and projected values. The capacity panel shows the illustrative return score for each expansion level.

In [ ]:
def visualize_scenario_analysis(scenarios: dict):
    """Plot the synthetic impacts and comparisons for the four scenarios."""
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle("Synthetic What-If Scenario Comparisons", fontsize=16, fontweight="bold")

    typhoon = scenarios["typhoon_disruption"]
    ax1.set_title("Typhoon Scenario: Modeled Impacts", fontweight="bold")
    categories = ["Capacity loss (%)", "Duration (hours)", "Cost impact ($K)"]
    values = [
        typhoon["capacity_reduction"] * 100,
        typhoon["duration_hours"],
        typhoon["estimated_cost_impact"] / 1000,
    ]
    bars = ax1.bar(categories, values, color=["#FF6B6B", "#FFD93D", "#FF8C00"])
    ax1.set_ylabel("Value in the unit named for each category")
    ax1.tick_params(axis="x", rotation=15)
    ax1.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                 f"{value:.1f}", ha="center", va="bottom")

    demand = scenarios["demand_surge"]
    ax2.set_title("Demand Surge: Mean Utilization", fontweight="bold")
    utilization_values = [
        demand["current_utilization"] * 100,
        demand["projected_utilization"] * 100,
    ]
    bars = ax2.bar(
        ["Current", "Projected"], utilization_values,
        color=["#4ECDC4", "#FF6B6B"],
    )
    ax2.set_ylabel("Utilization (%)")
    ax2.set_ylim(0, 105)
    ax2.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, utilization_values):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                 f"{value:.1f}%", ha="center", va="bottom")

    hub = scenarios["new_hub_addition"]
    ax3.set_title("New Hub Scenario: Efficiency Assumption", fontweight="bold")
    if "candidate_hub" in hub:
        efficiency_values = [hub["current_efficiency"] * 100, hub["projected_efficiency"] * 100]
        bars = ax3.bar(
            ["Current", "Assumed projected"], efficiency_values,
            color=["#95E77E", "#4ECDC4"],
        )
        ax3.set_ylabel("Modeled efficiency (%)")
        ax3.set_ylim(0, 105)
        ax3.tick_params(axis="x", rotation=10)
        ax3.grid(True, axis="y", alpha=0.3)
        for bar, value in zip(bars, efficiency_values):
            ax3.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                     f"{value:.1f}%", ha="center", va="bottom")
        ax3.text(
            0.5, 0.05, f"Candidate port: {hub['candidate_hub'] + 1}",
            transform=ax3.transAxes, ha="center",
        )
    else:
        ax3.text(0.5, 0.5, "No candidate available", ha="center", va="center",
                 transform=ax3.transAxes)

    expansion = scenarios["capacity_expansion"]
    ax4.set_title("Capacity Expansion: Illustrative ROI", fontweight="bold")
    if "expansion_options" in expansion:
        options = expansion["expansion_options"]
        percentages = [option["expansion_percentage"] * 100 for option in options]
        roi_values = [option["roi"] for option in options]
        ax4.scatter(percentages, roi_values, s=150, c="#4ECDC4", edgecolors="black", label="Option")
        recommended = expansion["recommended_option"]
        ax4.scatter(
            recommended["expansion_percentage"] * 100,
            recommended["roi"], s=280, facecolors="none", edgecolors="red",
            linewidths=2, label="Highest modeled ROI",
        )
        ax4.set_xlabel("Capacity expansion (%)")
        ax4.set_ylabel("Illustrative ROI score")
        ax4.grid(True, alpha=0.3)
        ax4.legend()
    else:
        ax4.text(0.5, 0.5, "No expansion options available", ha="center", va="center",
                 transform=ax4.transAxes)

    fig.tight_layout()
    plt.show()
    return fig


scenario_figure = visualize_scenario_analysis(scenarios)

## 6. Summarize the simulated history

The summary compares the means of the first five and final five recorded updates. These are descriptive comparisons within one stochastic simulation run; they do not isolate the effect of hub reconfiguration. The monitoring rules may not trigger, and a particular hub may not appear in the final configuration. That is a possible outcome of this simulation, not a reason to alter its inputs after observing the result.

In [ ]:
def analyze_digital_twin_performance(digital_twin: HubSpokeDigitalTwin) -> Optional[dict]:
    """Summarize descriptive changes and visualize recorded simulation metrics."""
    print("SIMULATION PERFORMANCE SUMMARY")
    metrics = digital_twin.historical_data["performance_metrics"]
    decisions = digital_twin.historical_data["optimization_decisions"]

    if len(metrics) < 2:
        print("Insufficient historical data for a trend comparison.")
        return None

    efficiency = [item["network_efficiency"] for item in metrics]
    service = [item["service_level"] for item in metrics]
    cost = [item["cost_per_container"] for item in metrics]
    window = min(5, len(metrics))

    initial_efficiency = float(np.mean(efficiency[:window]))
    final_efficiency = float(np.mean(efficiency[-window:]))
    initial_service = float(np.mean(service[:window]))
    final_service = float(np.mean(service[-window:]))
    initial_cost = float(np.mean(cost[:window]))
    final_cost = float(np.mean(cost[-window:]))

    efficiency_change = (
        (final_efficiency - initial_efficiency) / initial_efficiency * 100
        if initial_efficiency else 0.0
    )
    service_change = (
        (final_service - initial_service) / initial_service * 100
        if initial_service else 0.0
    )
    cost_change = (initial_cost - final_cost) / initial_cost * 100 if initial_cost else 0.0

    print(f"Recorded update intervals: {len(metrics)}")
    print(f"Descriptive efficiency change between windows: {efficiency_change:+.1f}%")
    print(f"Descriptive service-level change between windows: {service_change:+.1f}%")
    print(f"Descriptive cost change between windows: {cost_change:+.1f}%")
    print(f"Recorded hub reconfigurations: {len(decisions)}")
    if decisions:
        mean_estimate = float(np.mean([item["expected_improvement"] for item in decisions]))
        print(f"Mean illustrative improvement estimate: {mean_estimate:.1%}")

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    ax1, ax2, ax3, ax4 = axes.flat
    fig.suptitle("Descriptive Simulation History", fontsize=16, fontweight="bold")
    update_number = np.arange(1, len(metrics) + 1)

    ax1.set_title("Modeled Performance by Update", fontweight="bold")
    ax1.plot(update_number, efficiency, "o-", label="Efficiency", color="#FF6B6B", markersize=3)
    ax1.plot(update_number, service, "s-", label="Service level", color="#4ECDC4", markersize=3)
    ax1.set_xlabel("Update number")
    ax1.set_ylabel("Fraction")
    ax1.set_ylim(0, 1)
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.set_title("Modeled Cost Indicator by Update", fontweight="bold")
    ax2.plot(update_number, cost, "o-", color="#FFD93D", markersize=3, label="Cost per container")
    ax2.set_xlabel("Update number")
    ax2.set_ylabel("Modeled cost ($ per container)")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    ax3.set_title("Recorded Reconfiguration Estimates", fontweight="bold")
    if decisions:
        estimates = [item["expected_improvement"] * 100 for item in decisions]
        decision_number = np.arange(1, len(estimates) + 1)
        ax3.bar(decision_number, estimates, color="#95E77E", label="Illustrative estimate")
        ax3.axhline(np.mean(estimates), color="red", linestyle="--", label="Mean estimate")
        ax3.set_xlabel("Recorded decision number")
        ax3.set_ylabel("Illustrative estimate (%)")
        ax3.legend()
        ax3.grid(True, axis="y", alpha=0.3)
    else:
        ax3.text(0.5, 0.5, "No reconfigurations recorded", ha="center", va="center",
                 transform=ax3.transAxes)

    ax4.set_title("Descriptive First-Window / Final-Window Comparison", fontweight="bold")
    labels = ["Efficiency", "Service level", "Cost reduction"]
    values = [efficiency_change, service_change, cost_change]
    bars = ax4.bar(labels, values, color=["#FF6B6B", "#4ECDC4", "#FFD93D"])
    ax4.axhline(0, color="black", linewidth=0.8)
    ax4.set_ylabel("Relative change (%)")
    ax4.tick_params(axis="x", rotation=15)
    ax4.grid(True, axis="y", alpha=0.3)
    for bar, value in zip(bars, values):
        offset = 0.5 if value >= 0 else -0.5
        ax4.text(bar.get_x() + bar.get_width() / 2, value + offset,
                 f"{value:+.1f}%", ha="center",
                 va="bottom" if value >= 0 else "top")

    fig.tight_layout()
    plt.show()
    return {
        "efficiency_change_percent": efficiency_change,
        "service_change_percent": service_change,
        "cost_reduction_percent": cost_change,
        "recorded_reconfigurations": len(decisions),
        "recorded_updates": len(metrics),
    }


performance_results = analyze_digital_twin_performance(digital_twin)

## 7. Interpretation and limitations

This simulation illustrates an integrated sequence: synthetic data generation, state synchronization within the program, threshold-based decision support, scenario calculations, and descriptive monitoring. The sequence is useful for understanding how components of a digital-twin architecture can interact.

The implementation does not establish predictive accuracy, automated disruption detection performance, delivery-delay reductions, or financial savings. Its demand values are generated from a fixed synthetic baseline with a daily pattern and random noise; they are not learned forecasts. Vessel motion is a random displacement rather than navigation. The hub selection rule ranks available ports using utilization and random variation, while the reported improvement is randomly assigned. Scenario benefits and ROI use simplified assumptions. Accordingly, all KPI values and recommendations should be interpreted as outputs of an educational simulation, not as validated business results.

A production study would require reliable data definitions and units, calibrated models, explicit service and cost objectives, validation against historical or observed outcomes, and operational review of safety and capacity constraints. Those elements are outside this notebook's scope.